#Training GPT2 with REMI
This file is dedicated to Training GPT2 with REMI representation.

In [ ]:
#@title Downloading Required Packages
from IPython.display import clear_output
!pip install transformers
!pip install accelerate
!pip install datasets
!pip install evaluate
!pip install loguru
!pip install torchtoolkit
clear_output()
print("All packages imported.")

All packages imported.


In [ ]:
#@title Importing Required Libraries
from datasets import load_dataset

import transformers
from transformers import TrainingArguments, Trainer, AdamW
from transformers import GPT2LMHeadModel, GPT2Config

from transformers.utils import logging
from transformers.data.data_collator import DataCollatorMixin
import tensorboard
from tensorboard.plugins.hparams import api as hp

import torch
from torch import Tensor, LongTensor, stack, flip, cat, full, argmax
from torch.utils.data import Dataset, DataLoader
from torchtoolkit.data import create_subsets

from tensorflow.errors import FailedPreconditionError

from evaluate import load as load_metric

from tqdm import tqdm
from pathlib import Path
from typing import List, Dict, Any,Union
from loguru import logger as lg
import pickle
import json

from google.colab import drive


In [ ]:
#@title Configuration Parameters

#transformers.logging.set_verbosity_debug()

PAD_None = 0
BOS_None = 1
EOS_None = 2

TOKENS_PATH = (
    ".../ALL_TOKENS_REMI_augmented.pickle"
) # Importing the tokenized file(s).

MAX_SEQ_LEN=512
MIN_SEQ_LEN=128

TRAIN_EVAL_SPLIT_NUM=0.15

conf = GPT2Config(
    vocab_size=212,
    n_positions=768,
    n_embd=1024,
    n_layer=12,
    n_head=8,
    n_inner=2048,
    resid_pdrop=.1,
    embd_pdrop=.1,
    attn_pdrop=.1,
    padding_token_id=PAD_None,
    bos_token_id=BOS_None,
    eos_token_id=EOS_None,
    )

ARGS_SAVE_PATH = "..."
RESUME_FROM_CHECKPOINT=True

train_args = TrainingArguments(
    output_dir = ARGS_SAVE_PATH,
    overwrite_output_dir = False,
    do_train = True,
    do_eval = True,
    do_predict = False,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy = "steps",
    eval_accumulation_steps=None,

    max_steps=50000,
    warmup_steps=500,
    eval_steps=1000,
    save_steps=1000,
    logging_steps=1000,

    learning_rate=1e-4, 
    weight_decay=0.01,
    lr_scheduler_type="cosine_with_restarts",
    max_grad_norm=3.0,
    #warmup_ratio=0.3,

    log_level="debug",
    logging_strategy="steps",
    save_strategy="steps",
    save_total_limit=5,
    no_cuda=False,
    seed=444,
    load_best_model_at_end=True,
    label_smoothing_factor=0.,
    optim="adamw_torch",
    #report_to=["tensorboard"],
)



In [ ]:
#@title Some Classes and Functions
class MIDIDataset(Dataset):
    r"""Dataset for generator training
    :files_paths: can be a path to .json or .pickle file containing all the tokenized datas.
    :param files_paths: list of .json files to load. I'm not tokenizing here because I've already done it.
    """

    def __init__(
        self,
        files_paths: Union[str, Path],
        min_seq_len: int,
        max_seq_len: int,
        file_type: str = "json"
    ):
        self.min_seq_len = min_seq_len
        self.max_seq_len = max_seq_len
        self.file_type=file_type

        self.samples = self._create_samples(files_paths)

    def __getitem__(self, idx) -> Dict[str, torch.LongTensor]:
        return {"input_ids": self.samples[idx], "labels": self.samples[idx]}

    def __len__(self) -> int:
        return len(self.samples)

    def __repr__(self):
        return self.__str__()

    def __str__(self) -> str:
        return "No data loaded" if len(self) == 0 else f"{len(self.samples)} samples"

    def _create_samples(self, input_path: Union[str, Path]) -> List[torch.LongTensor]:
        """Used in __init__ to create the samples."""

        if isinstance(input_path, str):
            input_path = Path(input_path)

        samples = []
        if input_path.is_dir() == True:
            json_paths = list(input_path.glob("*.json"))
            for file_path in tqdm(
                json_paths, desc=f"Loading data: {input_path[0].parent}"
            ):
                with open(file_path) as json_file:
                    tokens = json.load(json_file)["ids"][0]  # first track
                i = 0
                while i < len(tokens):
                    if i >= len(tokens) - self.min_seq_len:
                        break  # last sample is too short
                    samples.append(torch.LongTensor(tokens[i : i + self.max_seq_len]))
                    i += len(samples[-1])  # could be replaced with max_seq_len

        else:  # if it's a single json file.
            with open(input_path, "rb") as file_path:
                # This single big file has everything. It's shape is List[List[int]]

                if self.file_type == "json":
                    all_data = json.load(file_path)
                elif self.file_type == "pickle":
                    all_data = pickle.load(file_path)
                else:
                    lg.critical(f"For file_type param, you have specified {self.file_type} but it should be 'json' or 'pickle'.")
                    raise Exception

            for tokens in all_data:
                i = 0
                while i < len(tokens):
                    if i >= len(tokens) - self.min_seq_len:
                        break  # last sample is too short
                    samples.append(torch.LongTensor(tokens[i : i + self.max_seq_len]))
                    i += len(samples[-1])  # could be replaced with max_seq_len

        return samples

def _pad_batch(
    examples: List[Dict[str, torch.LongTensor]], pad_token: int
) -> torch.LongTensor:
    """Collate `examples` into a batch, using the information in `tokenizer` for padding if necessary."""
    #lg.info(f"Examples: \n{examples}")

    # length_of_first = examples[0]["tokens"].size(0)
    length_of_first = examples[0]["input_ids"].size(0)

    # Check if padding is necessary.
    are_tensors_same_length = all(
        x["input_ids"].size(0) == length_of_first for x in examples
    )
    if are_tensors_same_length:
        return torch.stack([e["input_ids"] for e in examples], dim=0).long()
    else:
        # Creating the full tensor and filling it with our data.
        return torch.nn.utils.rnn.pad_sequence(
            [e["input_ids"] for e in examples], batch_first=True, padding_value=pad_token
        ).long()


class DataCollatorGen(DataCollatorMixin):
    def __init__(self, pad_token: int, return_tensors: str = "pt"):
        """Collator that simply pad the input sequences.
        Input_ids will be padded with the pad token given, while labels will be
        padded with -100.

        :param pad_token: pas token
        :param return_tensors:
        """
        self.pad_token = pad_token
        self.return_tensors = return_tensors

    def __call__(
        self, batch: List[Dict[str, Any]], return_tensors=None
    ) -> Dict[str, torch.LongTensor]:
        x, y = _pad_batch(batch, self.pad_token), _pad_batch(batch, -100)
        return {"input_ids": x, "labels": y}  # will be shifted in GPT2LMHeadModel forward


In [ ]:
#@title Creating training and evaluation datasets
#tokens_paths = list(Path(TOKENS_PATH).glob("*.json"))
dataset = MIDIDataset(
    TOKENS_PATH,
    max_seq_len=MAX_SEQ_LEN,
    min_seq_len=MIN_SEQ_LEN,
    file_type="pickle"
    )

subset_train, subset_valid = create_subsets(dataset, [TRAIN_EVAL_SPLIT_NUM])

In [ ]:
#@title Displaying some examples
lg.info(f"Lenghts of the train and valid datasets: {len(subset_train)} and {len(subset_valid)}")
lg.info(f"An example from subset_train: {subset_train[0]}")
lg.info(f"An example from subset_valid: {subset_valid[0]}")

lg.info(f"Configuration of GPT2LMHeadModel \n{conf}")

In [ ]:
#@title Initiating the model and custom metric functions
model = GPT2LMHeadModel(conf)
lg.info(f"Number of paramaters: {model.num_parameters()}")
lg.info(f"Models summary: \n {model}")

metrics = {metric: load_metric(metric) for metric in ["accuracy"]}

def compute_metrics(eval_pred):
    """Computes metrics for pretraining.
    Must use proprocess_logits function that converts logits to predictions (argmax or sampling).

    :param eval_pred: EvalPrediction containing predictions and labels
    :return: metrics
    """
    predictions, labels = eval_pred
    not_pad_mask = labels != -100
    labels, predictions = labels[not_pad_mask], predictions[not_pad_mask]
    return metrics["accuracy"].compute(predictions=predictions.flatten(), references=labels.flatten())

def preprocess_logits(logits: Tensor, _: Tensor) -> Tensor:
    """Preprocesses the logits before accumulating them during evaluation.
    This allows to significantly reduce the memory usage and make the training tractable.
    """
    pred_ids = argmax(logits, dim=-1)  # long dtype
    return pred_ids

In [ ]:
    #@title Initiating the Trainer
trainer = Trainer(
    model,
    args=train_args,
    train_dataset=subset_train,
    eval_dataset=subset_valid,
    data_collator=DataCollatorGen(pad_token=0),
    preprocess_logits_for_metrics=preprocess_logits,
    compute_metrics=compute_metrics,
)

In [ ]:
avoid_drive_error=3
for i in range(avoid_drive_error):
  try:
        train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
  except FailedPreconditionError:
        lg.error("Training stopped due to failed precondition error. Now mounting the drive again.")
        drive.mount('/content/drive', force_remount=True)
  except KeyboardInterrupt:
        lg.info("Training interrupted by user.")
        break

#lg.info("Now saving model and metrics.")
#trainer.save_model()
#trainer.log_metrics("train", train_result.metrics)
#trainer.save_metrics("train", train_result.metrics)
#trainer.save_state()